# 06. Evaluasi dan perbandingan model

Bandingkan struktur cluster, proporsi data ditugaskan, ukuran kelompok, interpretasi profil, dan kestabilan. Tidak memilih model hanya dari satu skor.

[Panduan](../00_panduan.ipynb) | [Sebelumnya](../05_dbscan/05_dbscan.ipynb) | [Berikutnya](../07_temporal_analysis/07_temporal_analysis.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. Perbandingan seluruh data dan subset identik](#s2)
- [3. Jalankan evaluasi](#s3)
- [4. Bandingkan karakteristik profil](#s4)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from src.common import ROOT, load_model, load_prepared, write_json
from src.modeling import cluster_metrics
from src.preprocessing import grid_panel

Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. Perbandingan seluruh data dan subset identik

Baris evaluasi pertama memakai seluruh training layak; DBSCAN menghitung metrik jarak tanpa noise. Baris kedua membatasi kedua metode pada subset training assigned DBSCAN yang sama. Silhouette tabel evaluasi memakai sampel tetap seed=42; tabel seleksi memakai rata-rata tiga seed. ARI adalah kesepakatan partisi, bukan akurasi.

In [2]:
def evaluate(root=ROOT):
    root = Path(root)
    cfg, _, train, X, _, _ = load_prepared(root)
    km, _ = load_model(root, "kmeans")
    db, dbmeta = load_model(root, "dbscan")
    rows = []
    for method, labels in [("kmeans", km.labels_), ("dbscan", db.labels_)]:
        rows.append({"model": method, "evaluation_scope": "seluruh baris training layak; noise dikeluarkan dari metrik jarak",
                     **cluster_metrics(X, labels, cfg["silhouette_sample_size"])})
    common = db.labels_ >= 0
    for method, labels in [("kmeans", km.labels_), ("dbscan", db.labels_)]:
        if common.any():
            rows.append({"model": method, "evaluation_scope": "subset identik: baris training yang ditugaskan DBSCAN",
                         **cluster_metrics(X[common], labels[common], cfg["silhouette_sample_size"])})
    scores = pd.DataFrame(rows)
    scores["dbscan_diagnostic_only"] = dbmeta["diagnostic_only"]
    scores.to_csv(root / "reports/evaluation/model_comparison.csv", index=False)
    ari = adjusted_rand_score(km.labels_[common], db.labels_[common]) if common.sum() > 1 else None
    write_json(root / "reports/evaluation/agreement.json", {"ari_common_assigned": ari,
               "n_common_assigned": int(common.sum()), "fraction_common_assigned": float(common.mean()),
               "meaning": "kesepakatan partisi, bukan akurasi terhadap ground truth"})
    return scores

<a id="s3"></a>
## 3. Jalankan evaluasi



In [3]:
comparison = evaluate(ROOT)
display(comparison)
import json
display(json.loads((ROOT / "reports/evaluation/agreement.json").read_text()))

,model,evaluation_scope,n_rows,n_clusters,n_assigned,noise_fraction,largest_cluster_fraction_assigned,smallest_cluster_size,silhouette,davies_bouldin,silhouette_n,dbscan_diagnostic_only
0,kmeans,seluruh baris training layak; noise dikeluarka...,13471,6,13471,0.000000,0.410363,536,0.398720,0.932800,2000,False
1,dbscan,seluruh baris training layak; noise dikeluarka...,13471,7,12253,0.090416,0.396638,28,0.126064,2.011581,2000,False
2,kmeans,subset identik: baris training yang ditugaskan...,12253,6,12253,0.000000,0.450257,352,0.425670,0.866523,2000,False
3,dbscan,subset identik: baris training yang ditugaskan...,12253,7,12253,0.000000,0.396638,28,0.126064,2.011581,2000,False


{'ari_common_assigned': 0.26652305388647424,
 'n_common_assigned': 12253,
 'fraction_common_assigned': 0.9095835498478212,
 'meaning': 'kesepakatan partisi, bukan akurasi terhadap ground truth'}

<a id="s4"></a>
## 4. Bandingkan karakteristik profil

Nama deskriptif disusun setelah clustering pada notebook 04 dan 05. Tabel di sini memakai aturan nama yang sama. Kelompok A K-Means dan Kelompok A DBSCAN adalah kelompok berbeda; angka/huruf yang sama tidak menyamakan keduanya.

In [4]:
from src.mapping import profile_interpretation_table
for method in ["kmeans","dbscan"]:
    display(Markdown(f"### Kelompok {method.upper()}"))
    with pd.option_context("display.max_colwidth", None):
        display(profile_interpretation_table(ROOT, method, cfg))

### Kelompok KMEANS

,Kode algoritma,Nama untuk menjelaskan hasil,Gempa per kotak per triwulan (rata-rata),Magnitudo (rata-rata),Kedalaman km (rata-rata),Jumlah kotak-periode pembentuk kelompok
0,KMEANS_P0,"Kelompok A: Cenderung dangkal, magnitudo sekitar 4,2",1.69,4.21,23.28,2895
1,KMEANS_P1,"Kelompok B: Kedalaman sekitar 468 km, magnitudo 4,2",1.22,4.24,467.64,536
2,KMEANS_P2,"Kelompok C: Kedalaman sekitar 138 km, magnitudo 3,4",1.65,3.41,137.88,1348
3,KMEANS_P3,"Kelompok D: Kedalaman sekitar 145 km, magnitudo 4,2",2.41,4.23,145.44,1407
4,KMEANS_P4,"Kelompok E: Cenderung dangkal, magnitudo sekitar 3,4",1.68,3.35,19.59,5528
5,KMEANS_P5,Kelompok F: Kejadian lebih sering; cenderung dangkal,8.93,3.67,38.30,1757


### Kelompok DBSCAN

,Kode algoritma,Nama untuk menjelaskan hasil,Gempa per kotak per triwulan (rata-rata),Magnitudo (rata-rata),Kedalaman km (rata-rata),Jumlah kotak-periode pembentuk kelompok
0,DBSCAN_P0,"Kelompok A: Cenderung dangkal, magnitudo sekitar 3,7",1.00,3.66,18.53,4860
1,DBSCAN_P1,"Kelompok B: Kedalaman sekitar 144 km, magnitudo 3,9",2.91,3.93,144.34,459
2,DBSCAN_P2,"Kelompok C: Kedalaman sekitar 209 km, magnitudo 3,9",1.00,3.88,208.79,1730
3,DBSCAN_P3,Kelompok D: Kejadian lebih sering; cenderung dangkal,3.70,3.61,23.42,4538
4,DBSCAN_P4,"Kelompok E: Kedalaman sekitar 75 km, magnitudo 3,7",2.00,3.69,74.99,467
5,DBSCAN_P5,"Kelompok F: Kedalaman sekitar 95 km, magnitudo 3,6",3.32,3.63,95.03,171
6,DBSCAN_P6,"Kelompok G: Kedalaman sekitar 285 km, magnitudo 4,2",2.00,4.17,285.08,28
7,NOISE,Tidak masuk kelompok historis DBSCAN,8.09,3.97,165.10,1218


## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.